# Apache Spark from Groovy (classic mode)

Apache Spark 4.2 running *inside* the kernel as a `local[*]` driver — DataFrames,
Spark SQL, and user-defined functions, all via `@Grab`. Requirements: the kernel
must run on **JDK 17, 21 or 25** (Spark 4.2 added Java 25 support), and the first
run of the next cell downloads Spark's ~300 MB dependency tree (cached afterwards).

**Kernel flags**: a Spark driver normally gets a dozen `--add-opens` flags from
`spark-class`; an in-kernel driver must carry them itself or JDK module
encapsulation bites (e.g. `InaccessibleObjectException ... does not "opens
java.io"` from Spark's serialization machinery). Install the kernelspec with
Spark's own canonical list — it leads with `-XX:+IgnoreUnrecognizedVMOptions`,
so it is safe across JDK versions:

```
./gradlew installKernelSpec -PjavaHome=/path/to/jdk -PjvmArgs="$(
  java -cp $SPARK_HOME/jars/spark-launcher_2.13-4.2.0.jar:<groovy-jar> \
    groovy.ui.GroovyMain -e 'print org.apache.spark.launcher.JavaModuleOptions.defaultModuleOptions()'
) -Dorg.sparkproject.io.netty.tryReflectionSetAccessible=true -Dgroovy.jupyter.classOutputDir=$HOME/.groovy-jupyter/classes"
```

(or paste the list from the companion `spark-connect.ipynb`, which needs the
same flags).

Two idioms to know when driving Spark's Scala-flavoured API from Groovy:
use **method-call syntax** where Scala has no JavaBean getter (`spark.version()`,
`spark.conf()` — not `spark.version`), and prefer **classes over closures for
UDFs** (shown below).

For cluster use there is a second, thinner road — **Spark Connect** — covered in
its own notebook; and classic cluster mode needs the class-shipping recipe
described at the end.

In [1]:
@Grab('org.apache.spark:spark-sql_2.13:4.2.0')
import org.apache.spark.sql.SparkSession
spark = SparkSession.builder()
        .appName('groovy-jupyter')
        .master('local[*]')
        .config('spark.ui.enabled', 'false')
        .getOrCreate()
"Spark ${spark.version()} started with ${Runtime.runtime.availableProcessors()} local cores"

[Shell-0] INFO org.eclipse.aether.internal.impl.filter.PrefixesRemoteRepositoryFilterSource - Loaded 23628 auto-discovered prefixes for remote repository central (prefixes-central.txt)
[Shell-0] INFO org.eclipse.aether.internal.impl.filter.PrefixesRemoteRepositoryFilterSource - Loaded 23628 auto-discovered prefixes for remote repository gcs-maven-central-mirror (prefixes-gcs-maven-central-mirror.txt)
[Shell-0] INFO org.eclipse.aether.internal.impl.filter.PrefixesRemoteRepositoryFilterSource - Loaded 23628 auto-discovered prefixes for remote repository central (prefixes-central.txt)
[BfDependencyCollector-0-3] INFO org.eclipse.aether.internal.impl.filter.PrefixesRemoteRepositoryFilterSource - Loaded 23631 auto-discovered prefixes for remote repository sonatype-releases (prefixes-sonatype-releases.txt)
[BfDependencyCollector-0-3] INFO org.eclipse.aether.internal.impl.filter.PrefixesRemoteRepositoryFilterSource - Loaded 23631 auto-discovered prefixes for remote repository maven-central-re

Spark 4.2.0 started with 12 local cores

## DataFrames over the whisky dataset

Read the CSV sitting next to this notebook, register it as a SQL view, and note
the row-shaped results render as tables via `collectAsList()`:

In [2]:
whisky = spark.read()
        .option('header', true)
        .option('inferSchema', true)
        .csv('whiskey.csv')
whisky.createOrReplaceTempView('whisky')
"${whisky.count()} rows, ${whisky.columns().size()} columns"

26/08/28 17:32:36 INFO SharedState: Setting hive.metastore.warehouse.dir ('null') to the value of spark.sql.warehouse.dir.
26/08/28 17:32:36 INFO SharedState: Warehouse path is 'file:/Users/paulk/Projects/groovy-jupyter/notebooks/spark-warehouse'.
26/08/28 17:32:37 INFO InMemoryFileIndex: It took 9 ms to list leaf files for 1 paths.
26/08/28 17:32:37 INFO InMemoryFileIndex: It took 0 ms to list leaf files for 1 paths.
26/08/28 17:32:37 INFO FileSourceStrategy: Pushed Filters: 
26/08/28 17:32:37 INFO FileSourceStrategy: Post-Scan Filters: Set((length(trim(value#0, None)) > 0))
26/08/28 17:32:38 INFO CodeGenerator: Code generated in 69.733791 ms
26/08/28 17:32:38 INFO MemoryStore: MemoryStore started with capacity 9.4 GiB
26/08/28 17:32:38 INFO MemoryStore: Block broadcast_0 stored as values in memory (estimated size 376.0 B, free 9.4 GiB)
26/08/28 17:32:38 INFO MemoryStore: Block broadcast_0_piece0 stored as bytes in memory (estimated size 38.9 KiB, free 9.4 GiB)
26/08/28 17:32:38 INFO 

86 rows, 14 columns

In [3]:
asRows = { df -> df.collectAsList().collect { row ->
    (0..<row.size()).collectEntries { i -> [df.columns()[i], row.get(i)] } } }
asRows(spark.sql('''
    select Distillery, Smoky, Body, Medicinal
    from whisky
    order by Smoky desc, Body desc
    limit 5
'''))

26/08/28 17:32:39 INFO FileSourceStrategy: Pushed Filters: 
26/08/28 17:32:39 INFO FileSourceStrategy: Post-Scan Filters: Set()
26/08/28 17:32:39 INFO CodeGenerator: Code generated in 12.802125 ms
26/08/28 17:32:39 INFO CodeGenerator: Code generated in 3.383125 ms
26/08/28 17:32:39 INFO MemoryStore: Block broadcast_7 stored as values in memory (estimated size 376.0 B, free 9.4 GiB)
26/08/28 17:32:39 INFO MemoryStore: Block broadcast_7_piece0 stored as bytes in memory (estimated size 38.9 KiB, free 9.4 GiB)
26/08/28 17:32:39 INFO SparkContext: Created broadcast 7 from collectAsList at DirectMethodHandleAccessor.java:104
26/08/28 17:32:39 INFO FileSourceScanExec: Planning scan with bin packing, max size: 4194304 bytes, open cost is considered as scanning 4194304 bytes.
26/08/28 17:32:39 INFO SparkContext: Starting job: collectAsList at DirectMethodHandleAccessor.java:104
26/08/28 17:32:39 INFO DAGScheduler: Got job 4 (collectAsList at DirectMethodHandleAccessor.java:104) with 1 output pa

Distillery,Smoky,Body,Medicinal
Ardbeg,4,4,4
Laphroig,4,4,4
Lagavulin,4,4,4
Caol Ila,4,3,2
Talisker,3,4,3


## User-defined functions

The reliable recipe: a cell-defined class implementing `UDF1`/`UDF2` (and
`Serializable`). The session classloader makes the class visible to Spark's
in-process executors:

In [4]:
import org.apache.spark.sql.api.java.UDF2
import org.apache.spark.sql.types.DataTypes

class Intensity implements UDF2<Integer, Integer, Integer>, Serializable {
    Integer call(Integer smoky, Integer medicinal) { smoky * 2 + medicinal }
}
spark.udf().register('intensity', new Intensity(), DataTypes.IntegerType)
asRows(spark.sql('''
    select Distillery, intensity(Smoky, Medicinal) as peatiness
    from whisky
    order by peatiness desc
    limit 5
'''))

26/08/28 17:32:39 INFO FileSourceStrategy: Pushed Filters: 
26/08/28 17:32:39 INFO FileSourceStrategy: Post-Scan Filters: Set()
26/08/28 17:32:39 INFO CodeGenerator: Code generated in 2.433917 ms
26/08/28 17:32:39 INFO CodeGenerator: Code generated in 9.866292 ms
26/08/28 17:32:39 INFO MemoryStore: Block broadcast_9 stored as values in memory (estimated size 376.0 B, free 9.4 GiB)
26/08/28 17:32:39 INFO MemoryStore: Block broadcast_9_piece0 stored as bytes in memory (estimated size 38.9 KiB, free 9.4 GiB)
26/08/28 17:32:39 INFO SparkContext: Created broadcast 9 from collectAsList at DirectMethodHandleAccessor.java:104
26/08/28 17:32:39 INFO FileSourceScanExec: Planning scan with bin packing, max size: 4194304 bytes, open cost is considered as scanning 4194304 bytes.
26/08/28 17:32:39 INFO SparkContext: Starting job: collectAsList at DirectMethodHandleAccessor.java:104
26/08/28 17:32:39 INFO DAGScheduler: Got job 5 (collectAsList at DirectMethodHandleAccessor.java:104) with 1 output par

Distillery,peatiness
Ardbeg,12
Lagavulin,12
Laphroig,12
Caol Ila,10
Clynelish,9


What about a Groovy closure coerced to the UDF interface? Let's find out —
honestly — what happens:

In [5]:
import org.apache.spark.sql.api.java.UDF1
closureUdf = { x -> x + 1 } as UDF1
try {
    spark.udf().register('inc', closureUdf, DataTypes.IntegerType)
    asRows(spark.sql('select Distillery, inc(Body) as bodyPlus from whisky limit 3'))
} catch (Throwable t) {
    def root = t
    while (root.cause != null) root = root.cause
    "closure UDF failed as expected — ${root.class.simpleName}: ${root.message?.take(120)} " +
    '(closures capture the non-serializable script owner; use a small class as above, ' +
    'or a @CompileStatic static method)'
}

26/08/28 17:32:39 INFO FileSourceStrategy: Pushed Filters: 
26/08/28 17:32:39 INFO FileSourceStrategy: Post-Scan Filters: Set()
26/08/28 17:32:39 INFO CodeGenerator: Code generated in 3.182583 ms
26/08/28 17:32:39 INFO MemoryStore: Block broadcast_11 stored as values in memory (estimated size 376.0 B, free 9.4 GiB)
26/08/28 17:32:39 INFO MemoryStore: Block broadcast_11_piece0 stored as bytes in memory (estimated size 38.9 KiB, free 9.4 GiB)
26/08/28 17:32:39 INFO SparkContext: Created broadcast 11 from collectAsList at DirectMethodHandleAccessor.java:104
26/08/28 17:32:39 INFO FileSourceScanExec: Planning scan with bin packing, max size: 4194304 bytes, open cost is considered as scanning 4194304 bytes.
26/08/28 17:32:39 WARN SerializationDebugger: Cannot find private methods using reflection
java.lang.reflect.InaccessibleObjectException: Unable to make java.io.ObjectStreamClass$ClassDataSlot[] java.io.ObjectStreamClass.getClassDataLayout() throws java.io.InvalidClassException accessibl

closure UDF failed as expected — NotSerializableException: Cell5 (closures capture the non-serializable script owner; use a small class as above, or a @CompileStatic static method)

**But Groovy 6 changes this story**: *native lambdas* under `@CompileStatic`
are compiled with Java's serializable-lambda machinery (Spark's `UDF1` extends
`Serializable`) and capture **values**, not the enclosing script — so lambdas,
including capturing ones, serialize and run as UDFs. The condition is static
compilation; the identical syntax in a dynamic context becomes a closure-backed
proxy and fails as above:

In [ ]:
import groovy.transform.CompileStatic

@CompileStatic
class Lambdas {
    static UDF1<Integer, Integer> inc() {
        UDF1<Integer, Integer> f = (Integer i) -> i + 1        // non-capturing
        f
    }
    static UDF1<Integer, Integer> plus(int n) {
        UDF1<Integer, Integer> f = (Integer i) -> i + n        // captures a value
        f
    }
}
spark.udf().register('inc', Lambdas.inc(), DataTypes.IntegerType)
spark.udf().register('plus8', Lambdas.plus(8), DataTypes.IntegerType)
asRows(spark.sql('select Distillery, inc(Body) as bodyPlus, plus8(Smoky) as smokyPlus8 from whisky limit 3'))

## Cluster mode: the class-shipping recipe

Everything above runs in one JVM, so executors see cell-defined classes for
free. On a real cluster, executors are remote JVMs — they need the classes the
kernel compiles. Two supported routes:

1. **Classic:** start the kernel with the environment variable
   `GROOVY_JUPYTER_CLASS_DIR=/some/dir` (cell classes are then also written to
   disk) and build the session with
   `.config('spark.repl.class.outputDir', '/some/dir')` — Spark's driver serves
   that directory to executors, exactly as Apache Zeppelin's Spark interpreter
   does. Available since Spark 2.0.
2. **Spark Connect** (Spark's strategic thin-client direction): no in-kernel
   driver at all — see the companion Spark Connect notebook.

Finally, release the local session's resources:

In [6]:
spark.stop()
'Spark session stopped'

26/08/28 17:32:39 INFO SparkContext: SparkContext is stopping with exitCode 0 from stop at DirectMethodHandleAccessor.java:104.
26/08/28 17:32:39 INFO MapOutputTrackerMasterEndpoint: MapOutputTrackerMasterEndpoint stopped!
26/08/28 17:32:39 INFO MemoryStore: MemoryStore cleared
26/08/28 17:32:39 INFO BlockManager: BlockManager stopped
26/08/28 17:32:39 INFO BlockManagerMaster: BlockManagerMaster stopped
26/08/28 17:32:39 INFO OutputCommitCoordinator$OutputCommitCoordinatorEndpoint: OutputCommitCoordinator stopped!
26/08/28 17:32:39 INFO SparkContext: Successfully stopped SparkContext (Uptime: 3961 ms)


Spark session stopped